# Cache Line and False Sharing Mechanics

**Student:** Ainabek Aisultan  
**Student ID:** 230103254  
**GitHub:** ros1nant  
**Instructor:** Sufyan bin Uzayr

Run the cells in order. This notebook executes the complete CPU benchmark and generates
all required files in English, including Q1-Q5 based on the measurements from your runtime.

**Hardware requirement:** four available physical CPU cores, 64-byte cache lines, and
per-thread affinity. A GPU is not required. Some Colab runtimes expose fewer cores;
use a local Jupyter runtime or another suitable machine if the hardware check fails.

The repository also contains a separate MPI/CUDA notebook for the earlier lab.


## 1. Locate the repository and install dependencies

A local checkout is reused when available. Otherwise, this cell clones the specified
GitHub repository. Existing checkouts are not reset, pulled, or overwritten. Dependencies
are installed before importing Numba. The lab uses the Git handle `ros1nant` for seed binding.


In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json
import zipfile

REPOSITORY_URL = "https://github.com/ros1nant/mpi-cuda-stencil-ros1nant_230103254.git"
GIT_USERNAME = "ros1nant"
REPETITIONS = 3

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_DIR = next((p for p in candidates
                 if (p / "cache_line_false_sharing" / "run_lab.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path.cwd() / "mpi-cuda-stencil-ros1nant_230103254"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO_DIR)], check=True)
    if not (REPO_DIR / "cache_line_false_sharing" / "run_lab.py").is_file():
        raise RuntimeError("The existing checkout has no cache-line lab. Use a fresh checkout of main.")

LAB_DIR = REPO_DIR / "cache_line_false_sharing"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(LAB_DIR / "requirements.txt")], check=True)
print("Lab directory:", LAB_DIR)
print("Student: Ainabek Aisultan | ID: 230103254")


## 2. Check CPU topology and affinity prerequisites

This check identifies four distinct physical cores available to the process. It does not
replace unavailable hardware with synthetic measurements. A failing check means this runtime
cannot satisfy the experiment; switch to a suitable host before proceeding.


In [ ]:
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
from false_sharing_lab import init_system_context

context = init_system_context(GIT_USERNAME)
print(json.dumps(context, indent=2))
print("Hardware check passed. The full run uses four physical cores.")


check_code = "from false_sharing_lab import init_system_context; import json, sys; print(json.dumps(init_system_context(sys.argv[1]), indent=2))"
subprocess.run([sys.executable, "-c", check_code, GIT_USERNAME], cwd=LAB_DIR, check=True)
print("Hardware check passed. The full run uses four physical cores.")


In [ ]:
subprocess.run([
    sys.executable, str(LAB_DIR / "run_lab.py"),
    "--git-username", GIT_USERNAME,
    "--repeats", str(REPETITIONS),
], cwd=LAB_DIR, check=True)


## 4. Review measured results and answers Q1-Q5

The timings below are read from the newly generated manifest. Voluntary and involuntary
context switches are collected with `resource.getrusage` on Linux. On Windows those two
categories are unavailable and explicitly marked null; only the total is recorded.


In [ ]:
from IPython.display import display, Markdown

manifest = json.loads((LAB_DIR / "cache_lab_manifest.json").read_text(encoding="utf-8"))
assert manifest["verification_passed"] is True
assert manifest["iterations_per_thread"] == 60_000_000
assert manifest["num_threads"] == 4
print("verification_passed:", manifest["verification_passed"])
print(json.dumps(manifest["summary"], indent=2))
display(Markdown((LAB_DIR / "ANALYSIS.md").read_text(encoding="utf-8")))


## 5. Package and download the completed submission

The ZIP contains source code, the current measurements, Q1-Q5, LLVM IR, assembly,
and instructions. It excludes Git history, dependency environments, and scratch files.
On Colab the browser download starts automatically; in local Jupyter a file link is displayed.


In [ ]:
ARTIFACTS = [
    "false_sharing_lab.py", "run_lab.py", "generate_analysis.py", "verify_submission.py",
    "requirements.txt", "README.md", "ANALYSIS.md", "cache_lab_manifest.json",
    "worker_strided.ll", "worker_private.ll", "worker_strided_native.ll",
    "worker_private_native.ll", "worker_strided.asm", "worker_private.asm",
]
ZIP_PATH = REPO_DIR / "cache_line_false_sharing_230103254_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ARTIFACTS:
        source = LAB_DIR / name
        if not source.is_file():
            raise FileNotFoundError(source)
        archive.write(source, arcname=f"cache_line_false_sharing/{name}")
print("Submission ZIP:", ZIP_PATH)

try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(os.path.relpath(ZIP_PATH, Path.cwd())))
else:
    files.download(str(ZIP_PATH))


## 6. Save your new results to the repository

The notebook does not collect credentials or push automatically. From your own checkout,
copy in the generated lab files if you used a hosted runtime, then run:

```sh
git add cache_line_false_sharing
git commit -m "Record cache line false sharing benchmark results"
git push origin main
```

Share the repository link through the course submission channel after checking the report.
The existing repository is used at the student's request; the PDF specification suggests
the separate repository name `cache-line-false-sharing-ros1nant`.
